In [1]:
import os
import requests
import mne
from tqdm import tqdm
import zipfile
import numpy as np
import csv
import torch
import datetime
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

from torch.optim.lr_scheduler import ExponentialLR
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from scikeras.wrappers import KerasClassifier

import torchvision.transforms as transforms
from scipy.signal import welch, find_peaks
from scipy.linalg import hankel, eigh
from sklearn.decomposition import FastICA
from sklearn.preprocessing import MinMaxScaler
from PIL import Image
import matplotlib.pyplot as plt
from IPython import display as disp
from scipy.fftpack import fft, ifft  # Correct way
import warnings
from sklearn.cross_decomposition import CCA

from sklearn.decomposition import PCA

from scipy.signal import butter, filtfilt
from scipy.signal import detrend
print('Imports complete.')

2025-04-11 21:50:32.777134: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-11 21:50:32.791984: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744404632.809553 3005948 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744404632.814887 3005948 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-11 21:50:32.833362: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Imports complete.


In [2]:
def hurst_exponent(signal):
    N = len(signal)
    mean_signal = np.mean(signal)
    Y = np.cumsum(signal - mean_signal)
    R = np.max(Y) - np.min(Y)
    S = np.std(signal)

    if S == 0:
        return 0.5  # Neutral value if no variability
    return np.log(R / S) / np.log(N)

def higuchi_fd(signal, kmax=10):
    N = len(signal)
    L = []

    for k in range(1, kmax + 1):
        Lk = []
        for m in range(k):
            idx = np.arange(1, int(np.floor((N - m) / k)), dtype=int)
            Lmk = np.sum(np.abs(signal[m + idx * k] - signal[m + k * (idx - 1)]))
            norm = (N - 1) / (len(idx) * k)
            Lmk = (Lmk * norm) / k
            Lk.append(Lmk)
        L.append(np.mean(Lk))

    lnL = np.log(L)
    lnk = np.log(1.0 / np.arange(1, kmax + 1))
    return np.polyfit(lnk, lnL, 1)[0]
def tsallis_entropy(signal, q=2.0, bins=100):
    hist, _ = np.histogram(signal, bins=bins, density=True)
    hist = hist[hist > 0]  # Avoid log(0)

    if q == 1:
        return -np.sum(hist * np.log(hist))  # Shannon entropy
    else:
        return (1 - np.sum(hist ** q)) / (q - 1)
import numpy as np

def sample_entropy(signal, m=2, r=0.2):
    N = len(signal)
    r *= np.std(signal)

    def _phi(m):
        x = np.array([signal[i:i + m] for i in range(N - m + 1)])
        C = np.sum([np.sum(np.linalg.norm(x - x_i, axis=1) <= r) - 1 for x_i in x])
        return C / ((N - m + 1) * (N - m))

    return -np.log(_phi(m + 1) / _phi(m))


In [ ]:
save_dir = "CIF_MATRIX"
os.makedirs(save_dir, exist_ok=True)
new_files=['14']

output_path = 'Processed_EEG_Output'

# loop through all .npy files in the folder
for file in os.listdir(output_path):
    if file.endswith('.npy'):
        subject_id = file.split('_')[-1].split('.')[0]  # Extract subject ID from filename
        print(subject_id)
        if subject_id in new_files:
            print("found")
            subject_data = np.load(os.path.join(output_path, file), allow_pickle=True)

            sampling_rate = 200  # Hz
            time_duration = 120   # seconds
            target_data_points = sampling_rate * time_duration  

             # Define PCC matrix storage
            num_channels = 11  # Adjust based on your dataset
            num_videos = len(subject_data)  # Number of trials (videos)

            # Process each video trial
            for trial_data in subject_data:
                print(trial_data['rating'])
                if trial_data['rating'] < 0.5:  # Skip if rating is too low**
                    print(f"Skipping Subject {subject_id}, Video {trial_data['video_index']} due to low rating {trial_data['rating']}")
                    continue  # Skip this trial
                video_idx = trial_data['video_index']  # Load stored video index
                eeg_signals = trial_data['data']  # (channels, timepoints)
                 # Debug: Print video index and label before processing
                print(f"Processing: Subject {subject_id}, Video {video_idx}")
                print(f"Expected label: {trial_data['label']}")

                # Extract last 60 seconds of EEG data
                sampling_rate = 200  # Hz
                time_duration = 120   # seconds
                target_data_points = sampling_rate * time_duration  
                reshaped_data = eeg_signals[:, -target_data_points:]  # (11, 12000)

                raw_segments = np.array_split(reshaped_data, 40, axis=1)
                for segment_idx in range(40):
                    all_features = []
                    for ch1 in range(num_channels):
                            features = {
                                "Channel": ch1,
                                "SampleEntropy": sample_entropy(raw_segments[segment_idx][ch1, :]),
                                "TsallisEntropy": tsallis_entropy(raw_segments[segment_idx][ch1, :]),
                                "HiguchiFD": higuchi_fd(raw_segments[segment_idx][ch1, :]),
                                "HurstExponent": hurst_exponent(raw_segments[segment_idx][ch1, :])
                            }
                            all_features.append(features)
                    np.save(os.path.join(save_dir, f"subject_{subject_id}_video_{video_idx}_segment_{segment_idx}_features.npy"), all_features)
                     # Also save the emotion label associated with this segment (same label for each segment in a video)
                    label_dict = {segment_idx: trial_data['label']}  # Store the same label for each segment
                    
                    # Debug: Verify label before saving
                    print(f"Saving label: {label_dict} at subject_{subject_id}_video_{video_idx}_segment_{segment_idx}_label.npy")

                    np.save(os.path.join(save_dir, f"subject_{subject_id}_video_{video_idx}_segment_{segment_idx}_label.npy"), label_dict)
                print(f"Saved feature matrices for subject {subject_id}, trial {video_idx}")
            print(f"Saved feature matrices for subject {subject_id}")

3
4
18
16
11
5
2
10
17
19
14
found
0.5
Processing: Subject 14, Video 41
Expected label: Happy
Saving label: {0: 'Happy'} at subject_14_video_41_segment_0_label.npy
Saving label: {1: 'Happy'} at subject_14_video_41_segment_1_label.npy
Saving label: {2: 'Happy'} at subject_14_video_41_segment_2_label.npy
Saving label: {3: 'Happy'} at subject_14_video_41_segment_3_label.npy
Saving label: {4: 'Happy'} at subject_14_video_41_segment_4_label.npy
Saving label: {5: 'Happy'} at subject_14_video_41_segment_5_label.npy
Saving label: {6: 'Happy'} at subject_14_video_41_segment_6_label.npy
Saving label: {7: 'Happy'} at subject_14_video_41_segment_7_label.npy
Saving label: {8: 'Happy'} at subject_14_video_41_segment_8_label.npy
Saving label: {9: 'Happy'} at subject_14_video_41_segment_9_label.npy
Saving label: {10: 'Happy'} at subject_14_video_41_segment_10_label.npy
Saving label: {11: 'Happy'} at subject_14_video_41_segment_11_label.npy
Saving label: {12: 'Happy'} at subject_14_video_41_segment_12_